# Notebook 16 -- Advanced Topics: Hamiltonian Simulation, HHL, and Beyond

This capstone notebook surveys the advanced features of the Goqu SDK.
We have spent fifteen notebooks building up from single qubits to
variational algorithms, noise models, transpilation, error mitigation,
and parameterized circuits. Now we bring it all together and explore the
frontiers: **Hamiltonian simulation**, the **HHL algorithm** for linear
systems, **Clifford simulation** at massive scale, **Pauli algebra**,
**pulse-level programming**, **operator representations**, **QASM/Quil
interop**, **backends**, and **observability hooks**.

By the end of this notebook you will be able to:

1. **Implement** Trotter-Suzuki Hamiltonian simulation and compare first vs second order.
2. **Describe** the HHL algorithm for solving linear systems.
3. **Demonstrate** Clifford simulation scaling to thousands of qubits.
4. **Explain** operator representations (Kraus, SuperOp, Choi, PTM) and convert between them.

### What we will cover

| Topic | Key idea |
|-------|----------|
| Trotter-Suzuki | Approximate $e^{-iHt}$ by slicing into small steps |
| HHL | Solve $Ax = b$ on a quantum computer |
| Clifford simulation | Stabilizer states up to 100,000 qubits |
| Pauli algebra | Multiply, commute, and compute expectation values |
| Pulse-level control | Program waveforms below the gate abstraction |
| Operator representations | Kraus, SuperOp, Choi, PTM -- and conversions |
| QASM / Quil interop | Emit and parse standard circuit formats |
| Backends | Local simulator, mock backend, cloud targets |
| Observability | Hook into simulation and transpilation events |

In [ ]:
import (
	"context"
	"fmt"
	"math"

	"github.com/splch/goqu/algorithm/hhl"
	"github.com/splch/goqu/algorithm/trotter"
	"github.com/splch/goqu/backend"
	"github.com/splch/goqu/backend/local"
	"github.com/splch/goqu/backend/mock"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/observe"
	"github.com/splch/goqu/pulse"
	"github.com/splch/goqu/pulse/waveform"
	qasmEmitter "github.com/splch/goqu/qasm/emitter"
	"github.com/splch/goqu/qasm/parser"
	quilEmitter "github.com/splch/goqu/quil/emitter"
	"github.com/splch/goqu/sim/clifford"
	"github.com/splch/goqu/sim/noise"
	"github.com/splch/goqu/sim/operator"
	"github.com/splch/goqu/sim/pauli"
	"github.com/splch/goqu/sim/statevector"
)

## Trotter-Suzuki Hamiltonian Simulation

Many quantum algorithms require simulating the time evolution of a
physical system whose dynamics are governed by a Hamiltonian $H$. The
time-evolution operator is

$$U(t) = e^{-iHt}$$

When $H$ is a sum of non-commuting terms $H = \sum_k H_k$, the matrix
exponential $e^{-i(H_1 + H_2 + \cdots)t}$ cannot be factored exactly.
The **Trotter-Suzuki decomposition** approximates it by splitting into
small time steps:

- **First-order (Lie-Trotter):**
  $e^{-iHt} \approx \left[\prod_k e^{-iH_k \Delta t}\right]^n$
  where $\Delta t = t/n$.

- **Second-order (Suzuki-Trotter):**
  $e^{-iHt} \approx \left[\prod_k e^{-iH_k \Delta t/2} \cdot \prod_{k'} e^{-iH_{k'} \Delta t/2}\right]^n$
  where the second product runs in reverse order. This symmetric
  splitting cancels the leading-order error term.

Let's simulate time evolution of a simple 2-qubit Heisenberg XX+ZZ
Hamiltonian: $H = X \otimes X + Z \otimes Z$.

In [ ]:
%%
ctx := context.Background()

// H = X*X + Z*Z (simple 2-qubit Hamiltonian)
xx, _ := pauli.Parse("XX")
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{xx, zz})

result, _ := trotter.Run(ctx, trotter.Config{
	Hamiltonian: ham,
	Time:        1.0,
	Steps:       4,
	Order:       trotter.First,
})

fmt.Printf("Trotter circuit: %d gates, depth %d\n",
	result.Circuit.Stats().GateCount, result.Circuit.Stats().Depth)
fmt.Println(draw.String(result.Circuit))

In [ ]:
%%
// Compare first-order vs second-order Trotter decompositions.
// More Trotter steps and higher order both improve accuracy at the
// cost of deeper circuits.
ctx := context.Background()
xx, _ := pauli.Parse("XX")
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{xx, zz})

r1, _ := trotter.Run(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 4, Order: trotter.First,
})
r2, _ := trotter.Run(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 4, Order: trotter.Second,
})

fmt.Printf("First-order:  %d gates, depth %d\n",
	r1.Circuit.Stats().GateCount, r1.Circuit.Stats().Depth)
fmt.Printf("Second-order: %d gates, depth %d\n",
	r2.Circuit.Stats().GateCount, r2.Circuit.Stats().Depth)

// Use Evolve to get final statevectors and compare fidelity.
sv1, _ := trotter.Evolve(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 4, Order: trotter.First,
}, nil)
sv2, _ := trotter.Evolve(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 4, Order: trotter.Second,
}, nil)

// High-precision reference (many steps).
svRef, _ := trotter.Evolve(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 100, Order: trotter.Second,
}, nil)

// Compute fidelity |<ref|approx>|^2.
fidelity := func(a, b []complex128) float64 {
	var dot complex128
	for i := range a {
		dot += complex(real(a[i]), -imag(a[i])) * b[i]
	}
	return real(dot)*real(dot) + imag(dot)*imag(dot)
}

fmt.Printf("\nFidelity vs high-precision reference:\n")
fmt.Printf("  First-order (4 steps):  %.8f\n", fidelity(svRef, sv1))
fmt.Printf("  Second-order (4 steps): %.8f\n", fidelity(svRef, sv2))
fmt.Println("\nSecond-order Trotter is more accurate at the same step count.")

## HHL Algorithm for Linear Systems

The **Harrow-Hassidim-Lloyd (HHL)** algorithm solves the linear system
$Ax = b$ by encoding the solution in the amplitudes of a quantum state.
For an $N \times N$ Hermitian matrix $A$ with eigenvalues $\lambda_j$
and eigenvectors $|u_j\rangle$, if we decompose $|b\rangle = \sum_j
\beta_j |u_j\rangle$, then the solution is $|x\rangle \propto \sum_j
(\beta_j / \lambda_j) |u_j\rangle$.

The algorithm uses three quantum registers:

1. **Ancilla** (1 qubit) -- flags successful eigenvalue inversion.
2. **Phase register** -- stores eigenvalue estimates via QPE.
3. **System register** -- encodes $|b\rangle$ and ultimately $|x\rangle$.

The success probability indicates how often post-selection succeeds.

In [ ]:
%%
ctx := context.Background()

// Simple 2x2 system: A = 0.5*Z (diagonal Hamiltonian).
// Eigenvalues are +0.5 and -0.5.
// |b> = H|0> = |+> = (|0> + |1>) / sqrt(2)
zi, _ := pauli.Parse("Z")
A, _ := pauli.NewPauliSum([]pauli.PauliString{zi.Scale(0.5)})
rhs, _ := builder.New("rhs", 1).H(0).Build()

result, err := hhl.Run(ctx, hhl.Config{
	Matrix:       A,
	RHS:          rhs,
	NumPhaseBits: 3,
	NumQubits:    1,
	Shots:        1000,
})
if err != nil {
	fmt.Println("Error:", err)
} else {
	fmt.Printf("HHL success probability: %.4f\n", result.Success)
	fmt.Printf("Solution state vector:   ")
	for i, amp := range result.StateVector {
		fmt.Printf("|%d>: %.4f+%.4fi  ", i, real(amp), imag(amp))
	}
	fmt.Println()
	fmt.Printf("\nHHL circuit: %d qubits, %d gates, depth %d\n",
		result.Circuit.NumQubits(),
		result.Circuit.Stats().GateCount,
		result.Circuit.Stats().Depth)
}

## Clifford Simulation -- Stabilizer States at Scale

Clifford circuits (composed of H, S, CNOT, and Pauli gates) can be
simulated efficiently on classical hardware using the **stabilizer
tableau** formalism. Where a full statevector simulator needs $2^n$
amplitudes, the Clifford simulator only needs $O(n^2)$ bits.

This makes it possible to simulate circuits with **thousands or even
100,000 qubits** -- far beyond any statevector simulator. The catch:
only Clifford gates are allowed (no T gate, no arbitrary rotations).

Let's create a 1000-qubit GHZ state: $|\text{GHZ}\rangle = (|0\rangle^{\otimes 1000} + |1\rangle^{\otimes 1000}) / \sqrt{2}$.

In [ ]:
%%
nQubits := 1000
b := builder.New("ghz-1000", nQubits)
b.H(0)
for i := 0; i < nQubits-1; i++ {
	b.CNOT(i, i+1)
}
b.MeasureAll()
c, err := b.Build()
if err != nil {
	fmt.Println("Build error:", err)
}

sim := clifford.New(nQubits)
counts, err := sim.Run(c, 100)
if err != nil {
	fmt.Println("Run error:", err)
}

// Should see only all-0s and all-1s.
fmt.Printf("1000-qubit GHZ state -- %d distinct outcomes from 100 shots:\n", len(counts))
for state, n := range counts {
	fmt.Printf("  |%s...%s> (%d qubits): %d shots\n",
		state[:3], state[len(state)-3:], len(state), n)
}
fmt.Println("\nA statevector simulator would need 2^1000 amplitudes for this.")
fmt.Println("The Clifford simulator handles it in milliseconds with O(n^2) memory.")

## Pauli Algebra

The Pauli operators $\{I, X, Y, Z\}$ form a basis for all $2 \times 2$
Hermitian matrices. Multi-qubit Pauli **strings** (tensor products like
$X \otimes Z \otimes I$) and their linear combinations (**Pauli sums**)
are the natural language for expressing Hamiltonians and observables.

Key algebraic properties:

- **Multiplication:** $X \cdot Y = iZ$, and the product of two Pauli
  strings is another Pauli string (up to a phase).
- **Commutation:** Two Pauli strings either commute or anticommute.
  They commute iff they anticommute at an even number of qubit positions.
- **Expectation values:** $\langle\psi|P|\psi\rangle$ can be computed in
  $O(2^n)$ time without materializing any matrix.

In [ ]:
%%
xx, _ := pauli.Parse("XX")
yy, _ := pauli.Parse("YY")
zz, _ := pauli.Parse("ZZ")

// Multiplication: XX * YY = ?
product := pauli.Mul(xx, yy)
fmt.Printf("XX * YY = %s\n", product)

// Commutation relations.
fmt.Printf("XX commutes with ZZ: %v\n", pauli.Commutes(xx, zz))
fmt.Printf("XX commutes with YY: %v\n", pauli.Commutes(xx, yy))
fmt.Printf("XY commutes with YX: ")
xy, _ := pauli.Parse("XY")
yx, _ := pauli.Parse("YX")
fmt.Printf("%v\n", pauli.Commutes(xy, yx))

// Expectation values on a Bell state.
sv := statevector.New(2)
bellC, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
sv.Evolve(bellC)

fmt.Printf("\nExpectation values for Bell state |Phi+>:\n")
fmt.Printf("  <Bell|XX|Bell> = %.4f\n", sv.ExpectPauliString(xx))
fmt.Printf("  <Bell|YY|Bell> = %.4f\n", sv.ExpectPauliString(yy))
fmt.Printf("  <Bell|ZZ|Bell> = %.4f\n", sv.ExpectPauliString(zz))

// PauliSum expectation (Hamiltonian).
ham, _ := pauli.NewPauliSum([]pauli.PauliString{
	xx,
	yy.Scale(-0.5),
	zz.Scale(0.3),
})
fmt.Printf("\n  <Bell| XX - 0.5*YY + 0.3*ZZ |Bell> = %.4f\n", sv.ExpectPauliSum(ham))

## Pulse-Level Programming

Gates are an abstraction. At the hardware level, quantum operations are
implemented by shaped microwave, laser, or RF **pulses** applied to
control lines (ports). Goqu's pulse package lets you program at this
lower level:

- **Port** -- a hardware I/O endpoint with a time resolution $dt$.
- **Frame** -- a software reference clock attached to a port, with a
  carrier frequency and phase.
- **Waveform** -- the pulse envelope (Gaussian, DRAG, constant, etc.).
- **Instructions** -- Play, Delay, ShiftPhase, SetFrequency, Capture, etc.

This is analogous to OpenPulse in Qiskit or calibration-level access
on IBM hardware.

In [ ]:
%%
// Define a drive port with dt = 0.2222 ns (typical for IBM hardware).
port := pulse.MustPort("d0", 2.222e-10)
frame := pulse.MustFrame("d0_frame", port, 5.0e9, 0.0)

// Build a Gaussian X pulse followed by a phase-shifted pulse.
gaussian := waveform.MustGaussian(0.5, 160e-9, 40e-9)

prog, err := pulse.NewBuilder("x-pulse").
	AddPort(port).
	AddFrame(frame).
	Play(frame, gaussian).
	Delay(frame, 100e-9).
	ShiftPhase(frame, math.Pi/2).
	Play(frame, gaussian).
	Capture(frame, 1000e-9).
	Build()

if err != nil {
	fmt.Println("Error:", err)
} else {
	stats := prog.Stats()
	fmt.Printf("Pulse program: %d ports, %d frames, %d instructions\n",
		stats.NumPorts, stats.NumFrames, stats.NumInstructions)
	fmt.Printf("Total duration: %.2f ns\n", stats.TotalDuration*1e9)
	fmt.Println(prog)
}

## Operator Representations

A quantum channel can be described in four equivalent ways, each useful
for different purposes:

| Representation | Definition | Use case |
|:--|:--|:--|
| **Kraus** | $\mathcal{E}(\rho) = \sum_k E_k \rho E_k^\dagger$ | Simulation, physical intuition |
| **SuperOp** | $\text{vec}(\mathcal{E}(\rho)) = S \cdot \text{vec}(\rho)$ | Composition (matrix multiply) |
| **Choi** | $\Lambda = (\mathcal{I} \otimes \mathcal{E})(|\Omega\rangle\langle\Omega|)$ | CP checks (positive semidefinite?) |
| **PTM** | $R_{ij} = \text{Tr}(\sigma_i \cdot \mathcal{E}(\sigma_j)) / d$ | Pauli error analysis |

Goqu provides conversion functions between all representations.

In [ ]:
%%
// Start from a depolarizing noise channel and convert between representations.
ch := noise.Depolarizing1Q(0.1)
kraus := operator.FromChannel(ch)

superOp := operator.KrausToSuperOp(kraus)
choi := operator.KrausToChoi(kraus)
ptm := operator.KrausToPTM(kraus)

fmt.Printf("Depolarizing(p=0.1) representations:\n")
fmt.Printf("  Kraus operators:    %d\n", len(kraus.Operators()))
fmt.Printf("  SuperOp matrix:     %d elements (4x4)\n", len(superOp.Matrix()))
fmt.Printf("  Choi matrix:        %d elements (4x4)\n", len(choi.Matrix()))
fmt.Printf("  PTM matrix:         %d elements (4x4)\n", len(ptm.Matrix()))

// Fidelity measures.
fmt.Printf("\nFidelity measures:\n")
fmt.Printf("  Process fidelity:      %.4f\n", operator.ProcessFidelity(kraus))
fmt.Printf("  Average gate fidelity: %.4f\n", operator.AverageGateFidelity(kraus))

// CPTP validation.
fmt.Printf("\nPhysicality check:\n")
fmt.Printf("  Is CPTP: %v\n", operator.IsCPTP(kraus, 1e-10))

// Round-trip: Kraus -> Choi -> Kraus and verify fidelity is preserved.
roundTrip := operator.ChoiToKraus(choi)
fmt.Printf("\nRound-trip (Kraus -> Choi -> Kraus):\n")
fmt.Printf("  Original process fidelity:    %.6f\n", operator.ProcessFidelity(kraus))
fmt.Printf("  Round-trip process fidelity:  %.6f\n", operator.ProcessFidelity(roundTrip))

## QASM and Quil Interop

Quantum circuits need to be exchanged between tools, compilers, and
hardware vendors. Goqu supports two standard formats:

- **OpenQASM 3.0** -- the IBM-originated standard used across the
  industry. `qasmEmitter.EmitString` produces QASM, and
  `parser.ParseString` reads it back.

- **Quil** -- Rigetti's instruction set. `quilEmitter.EmitString`
  produces Quil source.

Let's round-trip a circuit through OpenQASM 3.0 and also emit Quil.

In [ ]:
%%
c, _ := builder.New("qasm-demo", 2).H(0).CNOT(0, 1).MeasureAll().Build()

fmt.Println("Original circuit:")
fmt.Println(draw.String(c))

// Emit to OpenQASM 3.0.
qasmStr, err := qasmEmitter.EmitString(c)
if err != nil {
	fmt.Println("QASM emit error:", err)
} else {
	fmt.Println("OpenQASM 3.0:")
	fmt.Println(qasmStr)
}

// Parse back from QASM.
parsed, err := parser.ParseString(qasmStr)
if err != nil {
	fmt.Println("Parse error:", err)
} else {
	fmt.Println("Parsed back from QASM:")
	fmt.Println(draw.String(parsed))
}

In [ ]:
%%
// Emit the same circuit to Quil (Rigetti format).
c, _ := builder.New("quil-demo", 2).H(0).CNOT(0, 1).MeasureAll().Build()

quilStr, err := quilEmitter.EmitString(c)
if err != nil {
	fmt.Println("Quil emit error:", err)
} else {
	fmt.Println("Quil:")
	fmt.Println(quilStr)
}

## Backends

The `backend.Backend` interface abstracts over execution targets:

```go
type Backend interface {
    Name() string
    Target() target.Target
    Submit(ctx, req) (*Job, error)
    Status(ctx, jobID) (*JobStatus, error)
    Result(ctx, jobID) (*Result, error)
    Cancel(ctx, jobID) error
}
```

Goqu ships with:

- **`local.Backend`** -- runs circuits on the local statevector simulator.
- **`mock.Backend`** -- a configurable test double for job pipelines.
- Cloud backends for IBM, Google, IonQ, Quantinuum, Rigetti, and AWS Braket.

All backends share the same submit/status/result lifecycle, so code
written against one backend works with any other.

In [ ]:
%%
// Local backend: runs circuits on the local statevector simulator.
lb := local.New()
fmt.Printf("Local backend: %q\n", lb.Name())

c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).MeasureAll().Build()
ctx := context.Background()

job, err := lb.Submit(ctx, &backend.SubmitRequest{
	Circuit: c,
	Shots:   1000,
	Name:    "bell-test",
})
if err != nil {
	fmt.Println("Submit error:", err)
} else {
	fmt.Printf("Job submitted: ID=%s, State=%s\n", job.ID, job.State)

	res, err := lb.Result(ctx, job.ID)
	if err != nil {
		fmt.Println("Result error:", err)
	} else {
		fmt.Printf("Results (%d shots):\n", res.Shots)
		for bitstring, count := range res.ToCounts() {
			fmt.Printf("  |%s>: %d\n", bitstring, count)
		}
	}
}

// Mock backend: configurable test double.
mb := mock.New("test-backend")
fmt.Printf("\nMock backend: %q\n", mb.Name())

## Observability Hooks

When running quantum workloads in production, you need visibility into
what is happening: how long simulations take, how deep transpiled
circuits are, how many backend calls are made. Goqu's `observe` package
provides **zero-dependency hooks** that you attach to a `context.Context`.

Available hooks:

- `WrapSim` -- called around simulation execution.
- `WrapTranspile` -- called around the transpilation pipeline.
- `WrapPass` -- called around each individual transpilation pass.
- `WrapJob` -- called around job submission through result retrieval.
- `WrapHTTP` -- called around backend HTTP requests.
- `WrapSweep` -- called around parameter sweep execution.
- `OnJobPoll` -- called each time a job is polled for status.

This integrates naturally with OpenTelemetry or Prometheus via the
`otelbridge` and `prombridge` sub-packages.

In [ ]:
%%
hooks := &observe.Hooks{
	WrapSim: func(ctx context.Context, info observe.SimInfo) (context.Context, func(err error)) {
		fmt.Printf("[observe] Simulating %d-qubit circuit (%d gates, %d shots)...\n",
			info.NumQubits, info.GateCount, info.Shots)
		return ctx, func(err error) {
			if err == nil {
				fmt.Println("[observe] Simulation complete!")
			} else {
				fmt.Printf("[observe] Simulation failed: %v\n", err)
			}
		}
	},
}

ctx := observe.WithHooks(context.Background(), hooks)

// Now any simulation run with this context will trigger the hook.
// The local backend respects observe hooks.
lb := local.New()
c, _ := builder.New("observed", 2).H(0).CNOT(0, 1).MeasureAll().Build()
job, err := lb.Submit(ctx, &backend.SubmitRequest{Circuit: c, Shots: 100})
if err != nil {
	fmt.Println("Error:", err)
} else {
	res, _ := lb.Result(ctx, job.ID)
	fmt.Printf("\nGot %d shots with hooks active.\n", res.Shots)
}

fmt.Println("\nHooks are zero-cost when not attached (nil check).")
fmt.Println("In production, wire them to OpenTelemetry for distributed tracing.")

---

## Exercises

### Exercise 1 -- Trotter Accuracy Comparison

Sweep the number of Trotter steps from 1 to 16 for both first-order
and second-order decompositions of the Heisenberg XXZ Hamiltonian
$H = XX + YY + 0.5 \cdot ZZ$. Plot (or print) the fidelity vs step
count to see how second-order converges faster.

In [ ]:
%%
// Exercise 1: Trotter Accuracy Comparison
//
// Goal: Sweep Trotter steps from 1 to 16 for both first-order and
//       second-order decompositions and compare fidelity vs a reference.
//
// Hamiltonian: H = XX + YY + 0.5*ZZ (Heisenberg XXZ model)
//
// TODO: Build the Hamiltonian
// TODO: Compute a high-precision reference state
// TODO: Sweep step counts and print fidelity for both orders
//
// Skeleton:

ctx := context.Background()

// Step 1: Build the Hamiltonian
// xx, _ := pauli.Parse("XX")
// yy, _ := pauli.Parse("YY")
// zz, _ := pauli.Parse("ZZ")
// ham, _ := pauli.NewPauliSum([]pauli.PauliString{xx, yy, zz.Scale(???)})

// Step 2: Compute a high-precision reference (many steps, second-order)
// svRef, _ := trotter.Evolve(ctx, trotter.Config{
//     Hamiltonian: ???, Time: 1.0, Steps: 200, Order: trotter.Second,
// }, nil)

// Step 3: Define a fidelity function
// fidelity := func(a, b []complex128) float64 {
//     var dot complex128
//     for i := range a {
//         dot += complex(real(a[i]), -imag(a[i])) * b[i]
//     }
//     return real(dot)*real(dot) + imag(dot)*imag(dot)
// }

// Step 4: Sweep step counts and compare
// fmt.Printf("%6s  %16s  %16s\n", "Steps", "1st-order F", "2nd-order F")
// for _, steps := range []int{1, 2, 4, 8, 16} {
//     sv1, _ := trotter.Evolve(ctx, trotter.Config{
//         Hamiltonian: ???, Time: 1.0, Steps: ???, Order: trotter.First,
//     }, nil)
//     sv2, _ := trotter.Evolve(ctx, trotter.Config{
//         Hamiltonian: ???, Time: 1.0, Steps: ???, Order: trotter.Second,
//     }, nil)
//     fmt.Printf("%6d  %16.10f  %16.10f\n", steps, fidelity(svRef, sv1), fidelity(svRef, sv2))
// }

fmt.Println("TODO: Uncomment the code above, fill in the ???, and run!")

### Exercise 2 -- Custom Pulse Sequence for a Hadamard Gate

On superconducting hardware, the Hadamard gate $H = (X + Z)/\sqrt{2}$
can be decomposed as $H = RZ(\pi) \cdot RY(\pi/2)$. Build a pulse
sequence that implements this: a Y-rotation pulse (Gaussian on the
drive frame) followed by a Z-rotation (a virtual phase shift, which is
free on hardware).

In [ ]:
%%
// Exercise 2: Custom Pulse Sequence for a Hadamard Gate
//
// Goal: Build a pulse program that implements H = RZ(pi) . RY(pi/2).
//       RY is a physical Gaussian pulse; RZ is a virtual phase shift.
//
// TODO: Create a drive port and frame
// TODO: Build the pulse program with Play (for RY) and ShiftPhase (for RZ)
//
// Skeleton:

// Step 1: Define hardware port and frame
// port := pulse.MustPort("d0", 2.222e-10)
// frame := pulse.MustFrame("d0_frame", port, 5.0e9, 0.0)

// Step 2: Create the RY(pi/2) Gaussian pulse
// The amplitude is calibrated so the pulse area equals pi/2.
// ryPulse := waveform.MustGaussian(???, 160e-9, 40e-9)

// Step 3: Build the pulse program
// hadamardProg, err := pulse.NewBuilder("hadamard-pulse").
//     AddPort(???).
//     AddFrame(???).
//     Play(frame, ???).           // RY(pi/2) via physical pulse
//     ShiftPhase(frame, ???).     // RZ(pi) via virtual phase shift
//     Build()
//
// if err != nil {
//     fmt.Println("Error:", err)
// } else {
//     stats := hadamardProg.Stats()
//     fmt.Printf("Hadamard pulse program:\n")
//     fmt.Printf("  Instructions: %d\n", stats.NumInstructions)
//     fmt.Printf("  Total duration: %.1f ns\n", stats.TotalDuration*1e9)
//     fmt.Println(hadamardProg)
// }

fmt.Println("TODO: Uncomment the code above, fill in the ???, and run!")

## Key Takeaways -- Curriculum Conclusion

Over sixteen notebooks, we have built a comprehensive understanding of
quantum computing using the Goqu SDK:

1. **Hamiltonian simulation** via Trotter-Suzuki decomposition lets us
   approximate $e^{-iHt}$ for arbitrary Hamiltonians. Second-order
   Trotter converges faster than first-order at the same circuit depth.

2. **HHL** solves linear systems $Ax = b$ by encoding the solution in
   quantum amplitudes. It requires QPE, controlled rotations, and
   post-selection.

3. **Clifford simulation** using stabilizer tableaux can handle up to
   100,000 qubits in polynomial time -- but only for Clifford circuits
   (H, S, CNOT, Pauli gates).

4. **Pauli algebra** provides the mathematical foundation for expressing
   Hamiltonians, computing expectation values, and analyzing commutation
   relations -- all in $O(2^n)$ without materializing matrices.

5. **Pulse-level programming** goes below the gate abstraction to
   control the actual microwave/laser signals. Virtual Z-rotations are
   free; physical pulses have finite duration and error.

6. **Operator representations** (Kraus, SuperOp, Choi, PTM) provide
   different views of the same quantum channel, each useful for different
   tasks: simulation, composition, physicality checks, and error analysis.

7. **QASM/Quil interop** enables circuit exchange across the quantum
   ecosystem. Round-trip parsing preserves circuit semantics.

8. **Backends** abstract over execution targets (local simulators, cloud
   hardware). The same code runs on any backend through the unified
   `Submit/Status/Result` interface.

9. **Observability hooks** provide production-grade visibility into
   quantum workloads without coupling to any specific metrics or tracing
   framework.

---

**Congratulations on completing the quantum computing curriculum!**
You now have the tools and understanding to build, simulate, transpile,
and run quantum algorithms -- from single-qubit gates all the way to
production-grade Hamiltonian simulation with observability.

The quantum advantage is not in any single algorithm, but in the ability
to compose all of these pieces: express a problem as a Hamiltonian,
simulate it with the right decomposition, transpile to hardware
constraints, mitigate noise, and monitor the entire pipeline.